# Raw SCADA cleaning rebuild

This notebook rebuilds a transparent, label-free SCADA branch from the challenge parquet files. The core cleaning path is:

```text
change-based events → per-turbine restoration → 15-min aggregate →
existing 30-min operating bin → optional RRS/physics diagnostics
```

Sections 1–6 are the reproducible cleaning branch. Sections 7–15 are optional parity, RRS, and physics diagnostics; they do not modify the cleaning branch or write a submission CSV. The existing `turbines_data.zip` is never overwritten.

### Data contract used here

| representation | treatment |
| --- | --- |
| missing channel value | no new update; carry the previous value forward |
| leading missing value | backfill only until the first observed update |
| linear signals | arithmetic mean within a time bin |
| direction signals | circular mean; retain resultant length `R` as concentration diagnostic |
| labels | reconstructed for calibration/audit only, never used by the cleaning mask |

This is an independent implementation of the data semantics used by the project. It does not copy predictions, hidden states, dates, or yaw values from another model.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

HERE = Path.cwd()
CANDIDATES = [
    HERE,
    HERE / "github_release",
    HERE.parent / "github_release",
]
REPO_ROOT = next(
    (p for p in CANDIDATES if (p / "src").exists() and (p.parent / "data").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate github_release next to data/.")
DATA_ROOT = REPO_ROOT.parent / "data"
OUTPUT_ROOT = REPO_ROOT / "cleaned_raw_v2"
SPLITS = ["train", "validate", "context", "test"]
SIGNALS = ["GenSpeed", "NacDir", "WindSpeed", "WindDir", "PitchAngle", "RotSpeed", "Power"]
print("repo:", REPO_ROOT)
print("data:", DATA_ROOT)
print("available splits:", [s for s in SPLITS if (DATA_ROOT / f"{s}.parquet").exists()])

repo: E:\EnergyHacks\github_release
data: E:\EnergyHacks\data
available splits: ['train', 'validate', 'context', 'test']


## 1. Read one turbine at a time

Reading by turbine keeps memory bounded for the high-frequency stream. The parquet files are queried with a turbine filter, so this notebook does not load the whole fleet into memory at once. The source parquet files are never overwritten.

In [2]:
def turbine_ids(path):
    ids = pq.read_table(path, columns=["turbine_id"]).column("turbine_id").unique()
    return sorted(ids.to_pylist())


def load_raw_turbine(path, turbine_id):
    columns = ["turbine_id", "ts", *SIGNALS, "yaw_misalignment_deg"]
    schema = pq.read_schema(path).names
    columns = [c for c in columns if c in schema]
    table = pq.read_table(
        path,
        columns=columns,
        filters=[("turbine_id", "=", turbine_id)],
    )
    frame = table.to_pandas()
    frame["ts"] = pd.to_datetime(frame["ts"])
    return frame.sort_values("ts").reset_index(drop=True)


for split in SPLITS:
    path = DATA_ROOT / f"{split}.parquet"
    if path.exists():
        print(split, turbine_ids(path))

train ['PPP_WTG12', 'PPP_WTG13', 'PPP_WTG14']
validate ['PPP_WTG17']
context ['PPP_WTG07', 'PPP_WTG08', 'PPP_WTG11', 'PPP_WTG15', 'PPP_WTG16', 'PPP_WTG18', 'PPP_WTG33', 'SSS_WTG04', 'SSS_WTG05', 'SSS_WTG07', 'SSS_WTG16']
test ['SSS_WTG06']


## 2. Reconstruct the change-based stream

A raw null is interpreted as "unchanged". After sorting by turbine and timestamp, each signal is restored independently. For a signal such as `NacDir`, `NacDir_update` marks rows that contain a newly transmitted value and `NacDir_age_sec` measures how long the carried-forward value has been reused. Leading values before the first update are backfilled only to make the initial interval usable; they should not be interpreted as an observed update. The update flags and ages are retained as diagnostics, but are not used to drop unchanged values.

In [3]:
def wrap180(values):
    return (np.asarray(values, dtype=float) + 180.0) % 360.0 - 180.0


def restore_stream(frame):
    # Never carry a value across turbines: each channel is restored within
    # its own turbine's timestamp-ordered stream.
    out = frame.sort_values(["turbine_id", "ts"]).reset_index(drop=True).copy()
    fill_cols = [c for c in SIGNALS if c in out.columns]
    for col in fill_cols:
        out[f"{col}_update"] = out[col].notna()
        last_update = out["ts"].where(out[col].notna())
        last_update = last_update.groupby(out["turbine_id"], sort=False).ffill()
        out[f"{col}_age_sec"] = (out["ts"] - last_update).dt.total_seconds()
    if fill_cols:
        # NaN means "unchanged since the previous update" in this
        # dataset. Match the reference preparation: ffill, then only
        # bfill leading values that precede the first update.
        grouped = out.groupby("turbine_id", sort=False)[fill_cols]
        out[fill_cols] = grouped.ffill()
        out[fill_cols] = out.groupby("turbine_id", sort=False)[fill_cols].bfill()
    if {"WindDir", "NacDir"}.issubset(out.columns):
        out["vane"] = wrap180(out["WindDir"] - out["NacDir"])
    out["date"] = out["ts"].dt.normalize()
    return out

## 3. Label-free operating filter

This cell reproduces the reference cleaning boundary. `NaN` means unchanged, so reconstructed rows are retained; no freshness or quantile filter is applied here. Operating filters are applied later by the existing `bin_turbine()` function, after the 15-minute aggregation.

The mask columns remain as all-true compatibility fields for the aggregation cell.

All thresholds are computed from the turbine's SCADA only and are recorded in the summary. In this reference branch the mask is intentionally all-true: this separates event-stream reconstruction from the later operating-condition filter. Applying a power or wind-speed filter before aggregation would mix two different questions and would make the cleaning comparison harder to interpret.

In [4]:
def clean_operating_rows(frame):
    """Prepare the reference cleaned stream without dropping valid repeats."""
    out = frame.copy()
    # NaN already means unchanged and was reconstructed in restore_stream.
    # The reference cleaning notebook aggregates every reconstructed row;
    # operating filters are applied later by bin_turbine().
    keep = pd.Series(True, index=out.index)
    out["clean_ok"] = keep
    out["power_ok"] = keep
    out["heading_ok"] = keep
    stats = {
        "input_rows": int(len(out)),
        "clean_rows": int(len(out)),
        "clean_fraction": 1.0 if len(out) else np.nan,
        "heading_rows": int(len(out)),
        "heading_fraction": 1.0 if len(out) else np.nan,
    }
    return out, stats

## 4. Robust 15-minute and 30-minute aggregates

The first output is a 15-minute reference aggregate: ordinary signals use arithmetic means and directions use circular means with resultant length $R$. The second output is produced by the existing `bin_turbine()` function, which applies the production operating filter and creates the 30-minute RRS input. Thus `15min` is the transparent reconstruction layer, while `30min` is the model-scale operating data. Labels, when present, are never used by the cleaning mask.

In [5]:
def circular_mean(series):
    x = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    z = np.mean(np.exp(1j * np.deg2rad(x)))
    return float(np.rad2deg(np.angle(z)) % 360.0)


def aggregate_clean(frame, freq="15min", mask_col="clean_ok"):
    x = frame.copy()
    x["bin"] = x["ts"].dt.floor(freq)
    input_n = x.groupby("bin", sort=True).size().rename("n_input")
    if mask_col not in x.columns:
        raise KeyError(f"Missing cleaning mask: {mask_col}")
    op = x[x[mask_col]].copy()
    if op.empty:
        return pd.DataFrame()
    g = op.groupby("bin", sort=True)
    # Match the reference workflow: arithmetic means for continuous
    # signals and circular means for the two direction measurements.
    linear = [c for c in ["GenSpeed", "WindSpeed", "PitchAngle", "RotSpeed", "Power"] if c in op.columns]
    out = g[linear].mean() if linear else pd.DataFrame(index=g.size().index)
    for direction in ("NacDir", "WindDir"):
        if direction not in op:
            continue
        radians = np.deg2rad(op[direction].astype(float))
        sin_mean = pd.Series(np.sin(radians), index=op.index).groupby(op["bin"]).mean()
        cos_mean = pd.Series(np.cos(radians), index=op.index).groupby(op["bin"]).mean()
        out[direction] = np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360.0
        # R = sqrt(mean(sin)^2 + mean(cos)^2): concentration of the
        # angles in this bin, not an additional direction measurement.
        out[f"{direction}_R"] = np.hypot(sin_mean, cos_mean)
    out["n_input"] = input_n.reindex(out.index)
    out["n_used"] = g.size()
    out["n_clean"] = out["n_used"]
    out["clean_fraction"] = out["n_used"] / out["n_input"].clip(lower=1)
    if "NacDir_update" in op:
        out["n_nac_updates"] = g["NacDir_update"].sum()
    for col in ("NacDir_age_sec", "WindDir_age_sec", "Power_age_sec"):
        if col in op:
            out[f"{col}_median"] = g[col].median()
    if "yaw_misalignment_deg" in x:
        labels = x.groupby(x["ts"].dt.normalize())["yaw_misalignment_deg"].median()
        out["yaw_misalignment_deg"] = labels.reindex(out.index.normalize()).to_numpy()
    if {"NacDir", "WindDir"}.issubset(out.columns):
        out["vane"] = wrap180(out["WindDir"] - out["NacDir"])
    out["turbine_id"] = x["turbine_id"].iloc[0]
    out["session"] = out.index
    out["timestamp"] = out.index
    out["date"] = out.index.normalize()
    out.index.name = "bin"
    return out.reset_index(drop=True)

## 5. Build the parallel cleaned branch

This cell writes only to `cleaned_raw_v2/15min` and `cleaned_raw_v2/30min`. The 15-minute output is the transparent reference intermediate; the 30-minute output reuses the production `bin_turbine()` path. It does not create a submission file and does not overwrite `turbines_data.zip`.

In [6]:
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
from fleet_context import FleetConfig, bin_turbine

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_rows = []
for split in SPLITS:
    path = DATA_ROOT / f"{split}.parquet"
    if not path.exists():
        continue
    for turbine_id in turbine_ids(path):
        raw = load_raw_turbine(path, turbine_id)
        restored = restore_stream(raw)
        cleaned, stats = clean_operating_rows(restored)
        # Reproduce the reference order: all reconstructed rows -> 15min
        # aggregate -> existing production 30min bin_turbine() filter.
        aggregated_15 = aggregate_clean(cleaned, "15min")
        out_dir_15 = OUTPUT_ROOT / "15min"
        out_dir_15.mkdir(parents=True, exist_ok=True)
        aggregated_15.to_parquet(out_dir_15 / f"{turbine_id}.parquet", index=False)

        aggregated_30 = bin_turbine(aggregated_15, FleetConfig()).reset_index()
        aggregated_30 = aggregated_30.rename(columns={"bin": "timestamp"})
        aggregated_30["turbine_id"] = turbine_id
        aggregated_30["session"] = aggregated_30["timestamp"]
        aggregated_30["date"] = pd.to_datetime(aggregated_30["timestamp"]).dt.normalize()
        aggregated_30["n_used"] = aggregated_30["n"]
        aggregated_30["n_clean"] = aggregated_30["n"]
        aggregated_30["n_input"] = aggregated_30["n"]
        aggregated_30["clean_fraction"] = 1.0
        out_dir_30 = OUTPUT_ROOT / "30min"
        out_dir_30.mkdir(parents=True, exist_ok=True)
        aggregated_30.to_parquet(out_dir_30 / f"{turbine_id}.parquet", index=False)
        summary_rows.append({"split": split, "turbine_id": turbine_id, **stats})
        print(f"{split:8s} {turbine_id:12s} {stats['input_rows']:,} -> {stats['clean_rows']:,}")
summary = pd.DataFrame(summary_rows)
summary

train    PPP_WTG12    5,740,015 -> 5,740,015
train    PPP_WTG13    5,766,997 -> 5,766,997
train    PPP_WTG14    5,652,506 -> 5,652,506
validate PPP_WTG17    5,732,985 -> 5,732,985
context  PPP_WTG07    5,646,934 -> 5,646,934
context  PPP_WTG08    5,594,844 -> 5,594,844
context  PPP_WTG11    5,704,689 -> 5,704,689
context  PPP_WTG15    5,791,406 -> 5,791,406
context  PPP_WTG16    5,857,769 -> 5,857,769
context  PPP_WTG18    5,794,192 -> 5,794,192
context  PPP_WTG33    5,695,667 -> 5,695,667
context  SSS_WTG04    4,783,127 -> 4,783,127
context  SSS_WTG05    4,762,621 -> 4,762,621
context  SSS_WTG07    4,771,792 -> 4,771,792
context  SSS_WTG16    4,710,053 -> 4,710,053
test     SSS_WTG06    4,665,730 -> 4,665,730


,split,turbine_id,input_rows,clean_rows,clean_fraction,heading_rows,heading_fraction
0,train,PPP_WTG12,5740015,5740015,1.0,5740015,1.0
1,train,PPP_WTG13,5766997,5766997,1.0,5766997,1.0
2,train,PPP_WTG14,5652506,5652506,1.0,5652506,1.0
3,validate,PPP_WTG17,5732985,5732985,1.0,5732985,1.0
4,context,PPP_WTG07,5646934,5646934,1.0,5646934,1.0
5,context,PPP_WTG08,5594844,5594844,1.0,5594844,1.0
6,context,PPP_WTG11,5704689,5704689,1.0,5704689,1.0
7,context,PPP_WTG15,5791406,5791406,1.0,5791406,1.0
8,context,PPP_WTG16,5857769,5857769,1.0,5857769,1.0
9,context,PPP_WTG18,5794192,5794192,1.0,5794192,1.0


## 6. Sanity checks before using the branch

The first comparison should be data-only: row counts, coverage, signal ages and directional resultant lengths. A resultant length `R` near 1 means the angles in a bin are concentrated; a low `R` means that the circular mean is weakly defined because directions span a wide arc. Only after this data-only check should the same frozen RRS parameters be run on the new 15-minute/30-minute files and compared with the existing zip on strict T3 LOTO.

In [7]:
for freq in ("15min", "30min"):
    files = sorted((OUTPUT_ROOT / freq).glob("*.parquet"))
    rows = []
    for file in files:
        available = pq.read_schema(file).names
        cols = [c for c in ["turbine_id", "clean_fraction", "NacDir_R", "WindDir_R", "n_clean"] if c in available]
        frame = pd.read_parquet(file, columns=cols)
        rows.append({
            "freq": freq,
            "turbine_id": file.stem,
            "bins": len(frame),
            "median_clean_fraction": frame.get("clean_fraction", pd.Series(dtype=float)).median(),
            "median_n_clean": frame.get("n_clean", pd.Series(dtype=float)).median(),
            "median_NacDir_R": frame.get("NacDir_R", pd.Series(dtype=float)).median(),
            "median_WindDir_R": frame.get("WindDir_R", pd.Series(dtype=float)).median(),
        })
    if rows:
        display(pd.DataFrame(rows).round(3))

,freq,turbine_id,bins,median_clean_fraction,median_n_clean,median_NacDir_R,median_WindDir_R
0,15min,PPP_WTG07,67836,1.0,83.0,0.999,0.997
1,15min,PPP_WTG08,67662,1.0,82.0,0.999,0.997
2,15min,PPP_WTG11,68371,1.0,83.0,0.999,0.996
3,15min,PPP_WTG12,68641,1.0,83.0,0.999,0.997
4,15min,PPP_WTG13,68286,1.0,84.0,0.999,0.997
5,15min,PPP_WTG14,67871,1.0,83.0,0.998,0.996
6,15min,PPP_WTG15,68466,1.0,85.0,0.999,0.997
7,15min,PPP_WTG16,68965,1.0,85.0,0.999,0.996
8,15min,PPP_WTG17,67198,1.0,85.0,0.999,0.998
9,15min,PPP_WTG18,68261,1.0,84.0,0.999,0.996


,freq,turbine_id,bins,median_clean_fraction,median_n_clean,median_NacDir_R,median_WindDir_R
0,30min,PPP_WTG07,23643,1.0,2.0,NaN,NaN
1,30min,PPP_WTG08,23602,1.0,2.0,NaN,NaN
2,30min,PPP_WTG11,22472,1.0,2.0,NaN,NaN
3,30min,PPP_WTG12,23226,1.0,2.0,NaN,NaN
4,30min,PPP_WTG13,23356,1.0,2.0,NaN,NaN
5,30min,PPP_WTG14,24692,1.0,2.0,NaN,NaN
6,30min,PPP_WTG15,23072,1.0,2.0,NaN,NaN
7,30min,PPP_WTG16,23390,1.0,2.0,NaN,NaN
8,30min,PPP_WTG17,23992,1.0,2.0,NaN,NaN
9,30min,PPP_WTG18,23526,1.0,2.0,NaN,NaN


## 7. Optional: frozen RRS parity audit

This is an audit of the cleaned-data branch, not the submission pipeline. It uses the new 30-minute files as the only SCADA input and keeps the RRS settings fixed. The core release form is

$$\hat y_i(d) = (C-\theta_i^\star) - c_{i,\lambda}(d),$$

where $C-\theta_i^\star$ is the absolute anchor and

$$c_{i,\lambda}(d)=c_{i,\mathrm{stable}}(d)+\lambda\left[c_{i,\mathrm{full}}(d)-c_{i,\mathrm{stable}}(d)\right].$$

Here $\lambda=0$ is the conservative stability-filtered amplitude, $\lambda=1$ is the full-amplitude endpoint, and the current release candidate uses $\lambda=0.75$. This notebook compares the unfiltered RRS stage, Stability-filtered RRS, and that fixed partial-amplitude version with the same constant-line baseline. No PELT threshold, neighbour rule, $B_0$ rule or amplitude parameter is tuned here. The release prediction fixes $\beta=1$; the LOTO audit below may fit a fold-specific global $\beta$ only to report a strict development comparison.

In [8]:
import sys
from dataclasses import replace
sys.path.insert(0, str(REPO_ROOT / "src"))
from yaw_model import (
    ModelConfig, estimate_theta_star, estimate_theta_star_from_vane, fit_and_predict, run_loto,
    apply_full_amplitude_stable_states, blend_stable_amplitude,
    build_turbine_bundle, apply_site_common_mode_veto, fit_global_beta,
    predict_from_bundle,
)
from fleet_context import FleetConfig, load_layout
from yaw_relative_state import RelativeStateConfig
from yaw_major_robustness import boundary_stability, apply_stability_filter

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
REQUESTED_TARGETS = ["PPP_WTG17", "SSS_WTG06"]
clean15_files = sorted((OUTPUT_ROOT / "15min").glob("*.parquet"))
clean30_files = sorted((OUTPUT_ROOT / "30min").glob("*.parquet"))
if not clean15_files or not clean30_files:
    raise FileNotFoundError("Run the cleaning cell first; expected cleaned_raw_v2/15min and 30min.")

def load_cleaned(files):
    result = {}
    for path in files:
        frame = pd.read_parquet(path)
        frame["timestamp"] = pd.to_datetime(frame["timestamp"])
        frame = frame.sort_values("timestamp").set_index("timestamp")
        if "n_clean" in frame:
            frame["n"] = frame["n_clean"]
        result[path.stem] = frame
    return result

clean15 = load_cleaned(clean15_files)
binned = load_cleaned(clean30_files)
available = sorted(binned)
audit_ids = [t for t in TRAIN + REQUESTED_TARGETS if t in binned and t in clean15]
target_ids = [t for t in REQUESTED_TARGETS if t in binned and t in clean15]
if not all(t in audit_ids for t in TRAIN):
    raise ValueError(f"Missing cleaned T3 turbine(s): {sorted(set(TRAIN) - set(audit_ids))}")

# Keep the existing estimator for the main A/B comparison.  This makes
# the cleaning comparison comparable with the current production model.
theta_star_legacy = {t: estimate_theta_star(clean15[t]) for t in audit_ids}
# The direct-vane estimator is retained as a diagnostic only; it changes
# the calibration observable and must not be mixed into the main result.
theta_star_direct = {t: estimate_theta_star_from_vane(clean15[t]) for t in audit_ids}
theta_star = theta_star_legacy
label_path = DATA_ROOT / "train.parquet"
labels = {}
for turbine in TRAIN:
    label_table = pq.read_table(
        label_path,
        columns=["turbine_id", "ts", "yaw_misalignment_deg"],
        filters=[("turbine_id", "=", turbine)],
    ).to_pandas()
    label_table = label_table.sort_values(["turbine_id", "ts"])
    # Labels are also change-based in the raw parquet: NaN means the
    # previous state persists. Reconstruct them before daily aggregation.
    label_table["yaw_misalignment_deg"] = (
        label_table.groupby("turbine_id")["yaw_misalignment_deg"].ffill()
    )
    label_table["yaw_misalignment_deg"] = (
        label_table.groupby("turbine_id")["yaw_misalignment_deg"].bfill()
    )
    label_table["date"] = pd.to_datetime(label_table["ts"]).dt.normalize()
    labels[turbine] = label_table.groupby("date")["yaw_misalignment_deg"].first().dropna()

layout = load_layout(REPO_ROOT, available)
location_ids = []
for site in ("PPP", "SSS"):
    location_file = DATA_ROOT / f"turbine_locations_{site}.csv"
    if location_file.exists():
        location_ids.extend(pd.read_csv(location_file)["turbine_id"].astype(str).tolist())
missing_fleet = sorted(set(location_ids) - set(available))
full_fleet_ready = not missing_fleet
FLEET = FleetConfig()
# Keep the production/Independent-Release RRS configuration for a fair parity check.
RELATIVE = RelativeStateConfig()
MODEL_CONFIG = ModelConfig()
print("cleaned turbines:", len(available), "| audit:", audit_ids, "| targets:", target_ids)
print("reconstructed labelled days:", {t: len(labels[t]) for t in TRAIN})
print("theta_star (legacy estimator used for A/B):", {t: round(v, 3) for t, v in theta_star_legacy.items()})
print("theta_star (direct-vane diagnostic):", {t: round(v, 3) for t, v in theta_star_direct.items()})
if not full_fleet_ready:
    print("WARNING: incomplete fleet; missing location turbines:", missing_fleet)
    print("RRS LOTO/target comparison is withheld until context.parquet and test.parquet are available.")

cleaned turbines: 16 | audit: ['PPP_WTG12', 'PPP_WTG13', 'PPP_WTG14', 'PPP_WTG17', 'SSS_WTG06'] | targets: ['PPP_WTG17', 'SSS_WTG06']
reconstructed labelled days: {'PPP_WTG12': 731, 'PPP_WTG13': 730, 'PPP_WTG14': 728}
theta_star (legacy estimator used for A/B): {'PPP_WTG12': -3.444, 'PPP_WTG13': 0.352, 'PPP_WTG14': -4.354, 'PPP_WTG17': -2.483, 'SSS_WTG06': -1.513}
theta_star (direct-vane diagnostic): {'PPP_WTG12': -3.448, 'PPP_WTG13': 0.369, 'PPP_WTG14': -4.389, 'PPP_WTG17': -2.494, 'SSS_WTG06': -2.488}


## 8. Optional: build the core label-free RRS stages

The production RRS path (pair consensus plus site common-mode veto) is the baseline. Six fixed perturbations are used only to decide whether a boundary is stable; the labels enter only in the next cell during LOTO calibration. If the raw context/test files are absent, this section withholds RRS metrics because a partial fleet cannot fairly test a neighbour-consensus model.

In [9]:
if not full_fleet_ready:
    baseline_clean = variant_clean = stable_clean = full_clean = partial_clean = {}
    stability_clean = pd.DataFrame()
    print("RRS stage construction skipped: raw context/test fleet is incomplete.")
else:
    def build_core_stage(fleet_cfg, relative_cfg):
        """Build the production RRS path on the complete cleaned fleet."""
        all_bundles = {
            turbine: build_turbine_bundle(
                turbine, binned, layout, fleet_cfg, relative_cfg, MODEL_CONFIG
            )
            for turbine in available
        }
        all_bundles = apply_site_common_mode_veto(all_bundles, MODEL_CONFIG)
        return {turbine: all_bundles[turbine] for turbine in audit_ids}

    baseline_clean = build_core_stage(FLEET, RELATIVE)
    VARIANTS = {
        "sector_20": (FLEET, replace(RELATIVE, sector_width_deg=20.0)),
        "sector_40": (FLEET, replace(RELATIVE, sector_width_deg=40.0)),
        "smooth_5": (FLEET, replace(RELATIVE, smooth_days=5)),
        "smooth_9": (FLEET, replace(RELATIVE, smooth_days=9)),
        "neighbour_tight": (replace(FLEET, radius_nn_factor=3.0, radius_site_factor=3.0, max_candidates=8), RELATIVE),
        "neighbour_broad": (replace(FLEET, radius_nn_factor=5.0, radius_site_factor=4.5, max_candidates=16), RELATIVE),
    }
    variant_clean = {
        name: build_core_stage(fleet_cfg, relative_cfg)
        for name, (fleet_cfg, relative_cfg) in VARIANTS.items()
    }
    stability_clean, _ = boundary_stability(
        baseline_clean, variant_clean, audit_ids, tolerance_days=14, min_support=4
    )
    if stability_clean.empty:
        stability_clean = pd.DataFrame(columns=["turbine", "baseline_date", "support_count", "support_fraction", "matched_dates", "supporting_variants", "stable"])
    stable_clean = apply_stability_filter(baseline_clean, stability_clean, MODEL_CONFIG)
    full_clean = apply_full_amplitude_stable_states(stable_clean, MODEL_CONFIG)
    partial_clean = blend_stable_amplitude(stable_clean, full_clean, 0.75)
    print("stable boundaries:", int(stability_clean["stable"].sum()) if not stability_clean.empty else 0)
    display(stability_clean.round(3))

stable boundaries: 5


C:\Users\HJL\AppData\Local\Temp\ipykernel_26140\3363887057.py:39: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(stability_clean.round(3))


,turbine,baseline_date,support_count,support_fraction,matched_dates,supporting_variants,stable
0,PPP_WTG13,2023-02-19,3,0.500,"2023-02-19, 2023-02-19, 2023-02-19","sector_20, smooth_9, neighbour_broad",False
1,PPP_WTG13,2023-07-27,4,0.667,"2023-07-27, 2023-07-28, 2023-07-27, 2023-07-27","sector_40, smooth_5, smooth_9, neighbour_broad",True
2,PPP_WTG17,2023-06-25,2,0.333,"2023-06-25, 2023-06-25","neighbour_tight, neighbour_broad",False
3,PPP_WTG17,2023-10-15,5,0.833,"2023-10-15, 2023-10-15, 2023-10-15, 2023-10-15...","sector_20, smooth_5, smooth_9, neighbour_tight...",True
4,PPP_WTG17,2024-01-07,6,1.000,"2024-01-06, 2024-01-06, 2024-01-06, 2024-01-06...","sector_20, sector_40, smooth_5, smooth_9, neig...",True
5,SSS_WTG06,2023-05-28,6,1.000,"2023-05-28, 2023-05-28, 2023-05-28, 2023-05-28...","sector_20, sector_40, smooth_5, smooth_9, neig...",True
6,SSS_WTG06,2023-09-24,5,0.833,"2023-10-01, 2023-09-24, 2023-09-24, 2023-09-24...","sector_20, smooth_5, smooth_9, neighbour_tight...",True
7,SSS_WTG06,2024-10-13,3,0.500,"2024-10-13, 2024-10-13, 2024-10-13","sector_40, neighbour_tight, neighbour_broad",False


## 9. Optional: compare constant-line, RRS and partial-amplitude RRS

`run_loto` fits $C$ and $\beta$ inside each held-out turbine fold. The constant-line scores are the same-fold baseline returned by that function. This is a development audit: the release model fixes $\beta=1$ and uses the fixed $\lambda=0.75$ candidate. No submission CSV is written.

In [10]:
if full_fleet_ready:
    STAGES = {
        "RRS_before_stability_filter": baseline_clean,
        "Stability_filtered_RRS": stable_clean,
        "Stability_filtered_RRS_lambda_0.75": partial_clean,
    }
else:
    STAGES = {}
    print("No RRS metrics reported: cleaned context/test fleet is incomplete.")
    constant_rows = []
    for holdout in TRAIN:
        fit_ids = [t for t in TRAIN if t != holdout]
        C_fit = float(np.mean([labels[t].mean() + theta_star[t] for t in fit_ids]))
        base = C_fit - theta_star[holdout]
        y = labels[holdout].dropna().to_numpy(dtype=float)
        err = base - y
        constant_rows.append({
            "holdout": holdout, "MAE": float(np.mean(np.abs(err))),
            "RMSE": float(np.sqrt(np.mean(err ** 2))), "base": base,
        })
    print("Constant-only cleaning audit (RRS withheld):")
    display(pd.DataFrame(constant_rows).round(3))
stage_rows = []
fold_rows = []
for name, bundles in STAGES.items():
    loto, macro = run_loto(TRAIN, bundles, labels, theta_star, MODEL_CONFIG)
    fold_rows.append(loto.assign(model=name))
    stage_rows.append({
        "model": name,
        "MAE": float(macro["mae"]),
        "RMSE": float(macro["rmse"]),
        "constant_MAE": float(macro["constant_mae"]),
        "constant_RMSE": float(macro["constant_rmse"]),
        "beta_mean": float(loto["beta"].mean()),
        "states_mean": float(loto["states"].mean()),
    })
display(pd.DataFrame(stage_rows).round(3))
if fold_rows:
    display(pd.concat(fold_rows, ignore_index=True).round(3))

target_rows = []
for name, bundles in STAGES.items():
    if not target_ids:
        continue
    C, beta, predictions, _ = fit_and_predict(
        TRAIN, target_ids, bundles, labels, theta_star, MODEL_CONFIG
    )
    for turbine in target_ids:
        pred = predictions[turbine]
        boundary_table = bundles[turbine]["boundaries"]
        active_col = "prediction_active" if "prediction_active" in boundary_table else "accepted"
        boundary_dates = pd.to_datetime(boundary_table.index[boundary_table[active_col].fillna(False)])
        target_rows.append({
            "model": name,
            "turbine": turbine,
            "C": C,
            "beta": beta,
            "states": int(pred["cluster"].nunique()),
            "boundaries": ", ".join(d.strftime("%Y-%m-%d") for d in boundary_dates),
            "prediction_min": float(pred["prediction"].min()),
            "prediction_max": float(pred["prediction"].max()),
        })
display(pd.DataFrame(target_rows).round(3))
print("Comparison complete; no CSV written.")

,model,MAE,RMSE,constant_MAE,constant_RMSE,beta_mean,states_mean
0,RRS_before_stability_filter,0.390,0.616,0.522,0.752,1.0,1.667
1,Stability_filtered_RRS,0.362,0.610,0.522,0.752,1.0,1.333
2,Stability_filtered_RRS_lambda_0.75,0.365,0.553,0.522,0.752,1.0,1.333


,holdout,mae,rmse,constant_mae,constant_rmse,beta,states,boundaries,model
0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none,RRS_before_stability_filter
1,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27",RRS_before_stability_filter
2,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none,RRS_before_stability_filter
3,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none,Stability_filtered_RRS
4,PPP_WTG13,0.522,0.961,1.004,1.387,1.0,2,2023-07-27,Stability_filtered_RRS
5,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none,Stability_filtered_RRS
6,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none,Stability_filtered_RRS_lambda_0.75
7,PPP_WTG13,0.533,0.789,1.004,1.387,1.0,2,2023-07-27,Stability_filtered_RRS_lambda_0.75
8,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none,Stability_filtered_RRS_lambda_0.75


,model,turbine,C,beta,states,boundaries,prediction_min,prediction_max
0,RRS_before_stability_filter,PPP_WTG17,-6.085,1.0,4,"2023-06-25, 2023-10-15, 2024-01-07",-6.680,-1.422
1,RRS_before_stability_filter,SSS_WTG06,-6.085,1.0,4,"2023-05-28, 2023-09-24, 2024-10-13",-8.162,-3.729
2,Stability_filtered_RRS,PPP_WTG17,-6.085,1.0,3,"2023-10-15, 2024-01-07",-6.345,-1.088
3,Stability_filtered_RRS,SSS_WTG06,-6.085,1.0,3,"2023-05-28, 2023-09-24",-8.301,-3.730
4,Stability_filtered_RRS_lambda_0.75,PPP_WTG17,-6.085,1.0,3,"2023-10-15, 2024-01-07",-7.905,0.317
5,Stability_filtered_RRS_lambda_0.75,SSS_WTG06,-6.085,1.0,3,"2023-05-28, 2023-09-24",-10.134,-3.330


Comparison complete; no CSV written.


## 10. Data parity audit against the existing cleaned zip

This is a data-only audit at the same 30-minute resolution. It compares direction/vane coverage and circular differences, then compares the resulting daily relative-heading signal for PPP17 and SSS06. No labels or predictions are used.

In [11]:
from baseline_loto_ridge import list_turbines, read_turbine
from fleet_context import bin_turbine
from yaw_relative_state import fleet_relative_series

zip_path = next((p for p in (REPO_ROOT.parent / 'turbines_data.zip', REPO_ROOT / 'turbines_data.zip') if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError('Existing turbines_data.zip was not found.')
zip_ids = list_turbines(str(zip_path))
parity_ids = sorted(set(zip_ids).intersection(available))
old_binned = {tid: bin_turbine(read_turbine(str(zip_path), tid), FleetConfig()) for tid in parity_ids}

def circular_abs_summary(values):
    # wrap180 may return a NumPy array; convert to a Series before dropna/quantiles.
    x = pd.Series(np.asarray(values).reshape(-1), dtype='float64')
    x = pd.to_numeric(x, errors='coerce').dropna().abs()
    return (float(x.median()), float(x.quantile(0.95))) if len(x) else (np.nan, np.nan)

parity_rows = []
for tid in parity_ids:
    old = old_binned[tid].copy()
    new = binned[tid].copy()
    old.index = pd.to_datetime(old.index)
    new.index = pd.to_datetime(new.index)
    columns = [c for c in ('NacDir', 'WindDir', 'vane') if c in old and c in new]
    aligned = old[columns].join(new[columns], how='inner', lsuffix='_old', rsuffix='_new')
    union_bins = len(old.index.union(new.index))
    row = {'turbine': tid, 'old_bins': len(old), 'new_bins': len(new), 'bin_overlap': len(aligned) / max(union_bins, 1)}
    for col in columns:
        med, q95 = circular_abs_summary(wrap180(aligned[f'{col}_new'] - aligned[f'{col}_old']))
        row[f'{col}_diff_med'] = med
        row[f'{col}_diff_q95'] = q95
    parity_rows.append(row)
display(pd.DataFrame(parity_rows).sort_values('turbine').round(3))

relative_rows = []
for tid in ('PPP_WTG17', 'SSS_WTG06'):
    if tid not in old_binned or tid not in binned:
        continue
    old_daily, old_diag, _ = fleet_relative_series(tid, old_binned, layout, FLEET, RELATIVE)
    new_daily, new_diag, _ = fleet_relative_series(tid, binned, layout, FLEET, RELATIVE)
    joined = old_daily[['relative_heading_smooth']].join(new_daily[['relative_heading_smooth']], how='inner', lsuffix='_old', rsuffix='_new').dropna()
    delta = pd.Series(
        np.asarray(wrap180(joined['relative_heading_smooth_new'] - joined['relative_heading_smooth_old'])).reshape(-1),
        index=joined.index, dtype='float64'
    ).dropna()
    relative_rows.append({
        'turbine': tid,
        'old_usable_pairs': int(old_diag.get('usable', pd.Series(dtype=bool)).sum()),
        'new_usable_pairs': int(new_diag.get('usable', pd.Series(dtype=bool)).sum()),
        'relative_overlap': len(joined) / max(len(old_daily.index.union(new_daily.index)), 1),
        'relative_diff_med': float(delta.abs().median()) if len(delta) else np.nan,
        'relative_diff_q95': float(delta.abs().quantile(0.95)) if len(delta) else np.nan,
    })
display(pd.DataFrame(relative_rows).round(3))


,turbine,old_bins,new_bins,bin_overlap,NacDir_diff_med,NacDir_diff_q95,WindDir_diff_med,WindDir_diff_q95,vane_diff_med,vane_diff_q95
0,PPP_WTG07,23643,23643,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,PPP_WTG08,23602,23602,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,PPP_WTG11,22472,22472,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,PPP_WTG12,23226,23226,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,PPP_WTG13,23356,23356,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,PPP_WTG14,24692,24692,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,PPP_WTG15,23072,23072,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,PPP_WTG16,23390,23390,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8,PPP_WTG17,23992,23992,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,PPP_WTG18,23526,23526,1.0,0.0,0.0,0.0,0.0,0.0,0.0


,turbine,old_usable_pairs,new_usable_pairs,relative_overlap,relative_diff_med,relative_diff_q95
0,PPP_WTG17,3,3,0.992,0.0,0.0
1,SSS_WTG06,4,4,0.866,0.0,0.0


## Appendix A. Exploratory Power and physics diagnostics

The following cells are research diagnostics, not part of the cleaning pipeline or final yaw model. Power is used only to examine operating-condition consistency and the SCADA-derived $\theta_i^\star$ reference. Boundaries, $B_0$, and $\beta$ remain fixed while diagnostic candidates $\lambda\in\{0.5,0.75,1.0\}$ and $k\in\{1.5,2,3\}$ are compared. Here $\lambda=1$ is only a diagnostic endpoint; it is not the release setting and does not generate a submission. No hidden target labels are used and no prediction is modified.

In [12]:
# 10. Frozen-boundary power-physics audit (diagnostic only)
# Power never changes segmentation or predictions here.
LAMBDA_POWER = (0.50, 0.75, 1.00)
K_POWER = (1.5, 2.0, 3.0)
POWER_WINDOW_DAYS = int(RELATIVE.change_window_days)

def _power_log_residual(frame):
    cols = ["WindSpeed", "WindDir", "Power"]
    if not set(cols).issubset(frame.columns):
        return pd.Series(dtype=float)
    x = frame[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    x = x[x["WindSpeed"].between(3.0, 13.0) & (x["Power"] > 0.0)]
    if x.empty:
        return pd.Series(index=frame.index, dtype=float)
    x["speed_bin"] = np.floor(x["WindSpeed"])
    x["sector_bin"] = np.floor((x["WindDir"] % 360.0) / 30.0)
    by_cell = x.groupby(["speed_bin", "sector_bin"])["Power"].transform("median")
    by_speed = x.groupby("speed_bin")["Power"].transform("median")
    expected = by_cell.fillna(by_speed).clip(lower=1e-6)
    return np.log((x["Power"] + 1e-6) / expected).rename("log_power_residual")

def _row_weighted_median(values, weights):
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    if values.ndim != 2 or values.shape[1] == 0:
        return np.full(values.shape[0], np.nan)
    valid = np.isfinite(values) & np.isfinite(weights)[None, :] & (weights[None, :] > 0)
    order = np.argsort(np.where(valid, values, np.inf), axis=1)
    ordered = np.take_along_axis(np.where(valid, values, np.inf), order, axis=1)
    w = np.take_along_axis(np.where(valid, weights[None, :], 0.0), order, axis=1)
    total = w.sum(axis=1)
    pos = (np.cumsum(w, axis=1) >= 0.5 * total[:, None]).argmax(axis=1)
    out = ordered[np.arange(len(values)), pos]
    out[total <= 0.0] = np.nan
    return out

def _relative_power(turbine):
    target = _power_log_residual(binned[turbine]).sort_index()
    diagnostics = stable_clean[turbine]["pair_diagnostics"]
    ids, weights = [], []
    for neighbour, row in diagnostics.iterrows():
        weight = float(row.get("reliability_weight", 0.0))
        if bool(row.get("usable", weight > 0.0)) and neighbour in binned and np.isfinite(weight) and weight > 0.0:
            ids.append(neighbour); weights.append(weight)
    if target.empty or not ids:
        return pd.Series(index=target.index, dtype=float), len(ids)
    nb = pd.concat({_id: _power_log_residual(binned[_id]) for _id in ids}, axis=1, sort=False).reindex(target.index)
    consensus = _row_weighted_median(nb.to_numpy(), weights)
    return target - pd.Series(consensus, index=target.index), len(ids)

def _window_median(series, date, side):
    x = pd.Series(series, dtype=float); x.index = pd.to_datetime(x.index)
    daily = x.groupby(x.index.normalize()).median()
    date = pd.Timestamp(date).normalize()
    if side == "pre":
        dates = pd.date_range(date - pd.Timedelta(days=POWER_WINDOW_DAYS), date - pd.Timedelta(days=1))
    else:
        dates = pd.date_range(date, date + pd.Timedelta(days=POWER_WINDOW_DAYS - 1))
    values = daily.reindex(dates).dropna()
    return float(values.median()) if len(values) >= 5 else np.nan

def _coverage(series, date):
    x = pd.Series(series, dtype=float); x.index = pd.to_datetime(x.index)
    daily = x.groupby(x.index.normalize()).median(); date = pd.Timestamp(date).normalize()
    dates = pd.date_range(date - pd.Timedelta(days=POWER_WINDOW_DAYS), date + pd.Timedelta(days=POWER_WINDOW_DAYS - 1))
    return float(daily.reindex(dates).notna().mean())

def _predicted_dlog_power(y_pre, y_post, k):
    if not np.isfinite(y_pre) or not np.isfinite(y_post):
        return np.nan
    angle = np.deg2rad(np.clip([y_pre, y_post], -89.0, 89.0))
    return float(k * (np.log(np.cos(angle[1])) - np.log(np.cos(angle[0]))))

if not full_fleet_ready:
    print("Power audit skipped: complete context/test fleet is required.")
else:
    C_power, beta_power = fit_global_beta(TRAIN, stable_clean, labels, theta_star, MODEL_CONFIG)
    candidates = {lam: blend_stable_amplitude(stable_clean, full_clean, lam) for lam in LAMBDA_POWER}
    predictions_power = {lam: {t: predict_from_bundle(t, candidates[lam][t], C_power, beta_power, theta_star) for t in audit_ids} for lam in LAMBDA_POWER}
    relative_power, meta = {}, {}
    for turbine in audit_ids:
        relative_power[turbine], meta[turbine] = _relative_power(turbine)
    rows = []
    for turbine in audit_ids:
        table = stable_clean[turbine]["boundaries"]
        active = "prediction_active" if "prediction_active" in table else "accepted"
        for date in (table.index[table[active].fillna(False)] if len(table) else []):
            date = pd.Timestamp(date).normalize()
            p0, p1 = _window_median(relative_power[turbine], date, "pre"), _window_median(relative_power[turbine], date, "post")
            obs = stable_clean[turbine]["observables"]
            h0, h1 = _window_median(obs["relative_heading_smooth"], date, "pre"), _window_median(obs["relative_heading_smooth"], date, "post")
            bg0, bg1 = _window_median(obs["background_residual_smooth"], date, "pre"), _window_median(obs["background_residual_smooth"], date, "post")
            event = table.loc[date]
            row = {"turbine": turbine, "boundary": date.date().isoformat(),
                   "heading_jump_deg": float(wrap180(h1 - h0)) if np.isfinite(h0) and np.isfinite(h1) else np.nan,
                   "pair_agreement": float(event.get("pair_agreement", np.nan)),
                   "observed_relative_log_power_change": float(p1 - p0) if np.isfinite(p0) and np.isfinite(p1) else np.nan,
                   "neighbour_background_change": float(bg1 - bg0) if np.isfinite(bg0) and np.isfinite(bg1) else np.nan,
                   "power_coverage": _coverage(relative_power[turbine], date), "usable_neighbours": meta[turbine]}
            for lam in LAMBDA_POWER:
                prediction = predictions_power[lam][turbine]["prediction"]
                y0, y1 = _window_median(prediction, date, "pre"), _window_median(prediction, date, "post")
                for k in K_POWER:
                    row[f"pred_dlogP_lam{lam:g}_k{k:g}"] = _predicted_dlog_power(y0, y1, k)
            rows.append(row)
    power_audit = pd.DataFrame(rows)
    display(power_audit.round(3))
    score_rows = []
    for lam in LAMBDA_POWER:
        for k in K_POWER:
            col = f"pred_dlogP_lam{lam:g}_k{k:g}"
            valid = power_audit[["observed_relative_log_power_change", col]].dropna()
            score_rows.append({"lambda": lam, "k": k, "n_boundaries": len(valid),
                               "mean_abs_log_power_gap": float((valid.iloc[:, 0] - valid.iloc[:, 1]).abs().mean()) if len(valid) else np.nan})
    display(pd.DataFrame(score_rows).round(3))
    print("Power audit only; no boundary or prediction was modified.")

,turbine,boundary,heading_jump_deg,pair_agreement,observed_relative_log_power_change,neighbour_background_change,power_coverage,usable_neighbours,pred_dlogP_lam0.5_k1.5,pred_dlogP_lam0.5_k2,pred_dlogP_lam0.5_k3,pred_dlogP_lam0.75_k1.5,pred_dlogP_lam0.75_k2,pred_dlogP_lam0.75_k3,pred_dlogP_lam1_k1.5,pred_dlogP_lam1_k2,pred_dlogP_lam1_k3
0,PPP_WTG13,2023-07-27,-3.521,0.884,0.039,0.681,1.000,4,0.007,0.009,0.014,0.008,0.011,0.017,0.010,0.013,0.020
1,PPP_WTG17,2023-10-15,-3.294,1.000,0.140,-0.576,0.976,3,-0.002,-0.002,-0.004,-0.002,-0.003,-0.005,-0.003,-0.004,-0.005
2,PPP_WTG17,2024-01-07,-9.448,1.000,0.261,-2.943,0.976,3,0.012,0.017,0.025,0.014,0.019,0.029,0.016,0.022,0.032
3,SSS_WTG06,2023-05-28,-8.254,1.000,0.111,0.979,1.000,4,-0.017,-0.023,-0.034,-0.020,-0.027,-0.040,-0.023,-0.031,-0.046
4,SSS_WTG06,2023-09-24,-7.106,1.000,0.170,0.072,1.000,4,0.018,0.024,0.036,0.021,0.028,0.042,0.024,0.032,0.048


,lambda,k,n_boundaries,mean_abs_log_power_gap
0,0.50,1.5,5,0.140
1,0.50,2.0,5,0.139
2,0.50,3.0,5,0.137
3,0.75,1.5,5,0.140
4,0.75,2.0,5,0.138
5,0.75,3.0,5,0.135
6,1.00,1.5,5,0.139
7,1.00,2.0,5,0.137
8,1.00,3.0,5,0.134


Power audit only; no boundary or prediction was modified.


## Appendix B. Physics-guided nuisance Ridge (diagnostic only)

This is not an ML yaw predictor. The candidate physics term $k\log\cos(\hat y_\lambda)$ is fixed in turn, while a small time-blocked Ridge model explains only wind-regime/background nuisance. It never creates a state, changes a boundary, or changes a prediction. Lower held-out residual means stronger diagnostic consistency, not a labelled yaw-accuracy score.

In [13]:
RIDGE_ALPHA = 10.0
N_TIME_BLOCKS = 6

def _daily_physics_frame(turbine, lam, k):
    power = relative_power[turbine]
    base = binned[turbine]
    ws = base["WindSpeed"].resample("D").median().rename("ws")
    wd = base["WindDir"].resample("D").median().rename("wd")
    bg = stable_clean[turbine]["observables"]["background_residual_smooth"].rename("bg")
    observed = power.groupby(power.index.normalize()).median().rename("observed")
    prediction = predictions_power[lam][turbine]["prediction"]
    angle = np.deg2rad(np.clip(prediction, -89.0, 89.0))
    physics = (k * np.log(np.cos(angle))).rename("physics")
    frame = pd.concat([observed, ws, wd, bg, physics], axis=1, sort=False).dropna()
    if frame.empty:
        return frame
    frame["sin_wd"] = np.sin(np.deg2rad(frame["wd"]))
    frame["cos_wd"] = np.cos(np.deg2rad(frame["wd"]))
    frame["ws2"] = frame["ws"] ** 2
    return frame

def _ridge_block_score(frame, columns=None):
    if columns is None:
        columns = ["ws", "ws2", "sin_wd", "cos_wd", "bg"]
    frame = frame.dropna(subset=["observed", "physics", *columns]).sort_index()
    if len(frame) < 60:
        return np.nan, np.nan, len(frame)
    blocks = np.minimum((np.arange(len(frame)) * N_TIME_BLOCKS) // len(frame), N_TIME_BLOCKS - 1)
    errors = []
    for block in range(N_TIME_BLOCKS):
        train = blocks != block; test = blocks == block
        if train.sum() < 20 or not test.any():
            continue
        x_train = frame.loc[train, columns].to_numpy(float)
        x_test = frame.loc[test, columns].to_numpy(float)
        mean, scale = x_train.mean(axis=0), x_train.std(axis=0)
        scale[scale == 0.0] = 1.0
        x_train = (x_train - mean) / scale
        x_test = (x_test - mean) / scale
        x_train = np.column_stack([np.ones(len(x_train)), x_train])
        x_test = np.column_stack([np.ones(len(x_test)), x_test])
        y_train = frame.loc[train, "observed"].to_numpy(float) - frame.loc[train, "physics"].to_numpy(float)
        penalty = np.eye(x_train.shape[1]) * RIDGE_ALPHA; penalty[0, 0] = 0.0
        coef = np.linalg.solve(x_train.T @ x_train + penalty, x_train.T @ y_train)
        prediction = frame.loc[test, "physics"].to_numpy(float) + x_test @ coef
        errors.extend((prediction - frame.loc[test, "observed"].to_numpy(float)).tolist())
    if not errors:
        return np.nan, np.nan, len(frame)
    errors = np.asarray(errors, float)
    return float(np.mean(np.abs(errors))), float(np.sqrt(np.mean(errors ** 2))), len(errors)

if not full_fleet_ready:
    print("Nuisance Ridge skipped: complete context/test fleet is required.")
else:
    ridge_rows = []
    for lam in LAMBDA_POWER:
        for k in K_POWER:
            fold_scores = [_ridge_block_score(_daily_physics_frame(t, lam, k)) for t in audit_ids]
            valid = [(mae, rmse, n) for mae, rmse, n in fold_scores if np.isfinite(mae) and np.isfinite(rmse)]
            ridge_rows.append({"lambda": lam, "k": k, "n_turbines": len(valid),
                              "n_days": int(sum(n for _, _, n in valid)),
                              "cv_mae": float(np.average([x[0] for x in valid], weights=[x[2] for x in valid])) if valid else np.nan,
                              "cv_rmse": float(np.sqrt(np.average([x[1] ** 2 for x in valid], weights=[x[2] for x in valid]))) if valid else np.nan})
    display(pd.DataFrame(ridge_rows).round(4))
    print("Physics-guided nuisance audit complete; no boundary or prediction was modified.")

,lambda,k,n_turbines,n_days,cv_mae,cv_rmse
0,0.50,1.5,5,3595,0.0774,0.1106
1,0.50,2.0,5,3595,0.0768,0.1098
2,0.50,3.0,5,3595,0.0755,0.1082
3,0.75,1.5,5,3595,0.0771,0.1103
4,0.75,2.0,5,3595,0.0764,0.1093
5,0.75,3.0,5,3595,0.0750,0.1075
6,1.00,1.5,5,3595,0.0768,0.1099
7,1.00,2.0,5,3595,0.0760,0.1089
8,1.00,3.0,5,3595,0.0745,0.1069


Physics-guided nuisance audit complete; no boundary or prediction was modified.


## Appendix C. Neighbour-only slow background diagnostic

The long sector baseline $$b^{long}_{ij,h}$$ stays fixed. We only smooth the existing neighbour--neighbour residual consensus over 60 or 90 days and compare its boundary change with the target signal. There is no new correction coefficient. Boundaries are frozen; this cell does not rerun PELT or change predictions.

In [14]:
SLOW_BG_WINDOWS = (60, 90)

if not full_fleet_ready:
    print("Slow-background diagnostic skipped: complete context/test fleet is required.")
else:
    from fleet_context import circular_median_deg

    def _slow_background(turbine, window):
        """Slow robust neighbour-only differential signal q_i(d)."""
        series = stable_clean[turbine]["observables"]["background_residual"].astype(float)
        return series.rolling(
            window, center=True, min_periods=max(14, window // 3)
        ).apply(circular_median_deg, raw=False).rename("q_slow")

    slow_background = {
        (t, window): _slow_background(t, window)
        for t in audit_ids
        for window in SLOW_BG_WINDOWS
    }
    slow_rows = []
    for turbine in audit_ids:
        obs = stable_clean[turbine]["observables"]
        fixed_signal = obs["relative_heading_smooth"].astype(float)
        table = stable_clean[turbine]["boundaries"]
        active = "prediction_active" if "prediction_active" in table else "accepted"
        dates = table.index[table[active].fillna(False)] if len(table) else []
        for date in dates:
            date = pd.Timestamp(date).normalize()
            fixed_jump = wrap180(_window_median(fixed_signal, date, "post") - _window_median(fixed_signal, date, "pre"))
            for window in SLOW_BG_WINDOWS:
                q = slow_background[(turbine, window)]
                q_pre = _window_median(q, date, "pre")
                q_post = _window_median(q, date, "post")
                q_jump = wrap180(q_post - q_pre) if np.isfinite(q_pre) and np.isfinite(q_post) else np.nan
                slow_rows.append({
                    "turbine": turbine, "boundary": date.date().isoformat(),
                    "window_days": window, "fixed_jump_deg": fixed_jump,
                    "q_jump_deg": q_jump,
                    "abs_q_to_target_jump": abs(q_jump) / max(abs(fixed_jump), 1e-6) if np.isfinite(q_jump) and np.isfinite(fixed_jump) else np.nan,
                })
    slow_boundary_audit = pd.DataFrame(slow_rows)
    display(slow_boundary_audit.round(3))
    print("Slow background is diagnostic only; fixed b, boundaries and predictions were unchanged.")

,turbine,boundary,window_days,fixed_jump_deg,q_jump_deg,abs_q_to_target_jump
0,PPP_WTG13,2023-07-27,60,-3.521,-0.064,0.018
1,PPP_WTG13,2023-07-27,90,-3.521,-0.312,0.089
2,PPP_WTG17,2023-10-15,60,-3.294,-0.247,0.075
3,PPP_WTG17,2023-10-15,90,-3.294,0.013,0.004
4,PPP_WTG17,2024-01-07,60,-9.448,-0.372,0.039
5,PPP_WTG17,2024-01-07,90,-9.448,-0.033,0.003
6,SSS_WTG06,2023-05-28,60,-8.254,-0.227,0.027
7,SSS_WTG06,2023-05-28,90,-8.254,-1.170,0.142
8,SSS_WTG06,2023-09-24,60,-7.106,-0.168,0.024
9,SSS_WTG06,2023-09-24,90,-7.106,-0.359,0.051


Slow background is diagnostic only; fixed b, boundaries and predictions were unchanged.


## Appendix D. Physics-guided Ridge with slow background (diagnostic only)

The slow neighbour-only signal is added only as a nuisance covariate. It is not used to alter $$b_{ij}$$, boundaries, or yaw predictions.

In [15]:
if not full_fleet_ready:
    print("Slow-background Ridge skipped: complete context/test fleet is required.")
else:
    ridge_slow_rows = []
    nuisance_columns = ["ws", "ws2", "sin_wd", "cos_wd", "bg", "slow_bg"]
    for window in SLOW_BG_WINDOWS:
        for lam in LAMBDA_POWER:
            for k in K_POWER:
                fold_scores = []
                for turbine in audit_ids:
                    frame = _daily_physics_frame(turbine, lam, k)
                    slow = slow_background[(turbine, window)].rename("slow_bg")
                    frame = frame.join(slow, how="inner").dropna()
                    fold_scores.append(_ridge_block_score(frame, columns=nuisance_columns))
                valid = [(mae, rmse, n) for mae, rmse, n in fold_scores if np.isfinite(mae) and np.isfinite(rmse)]
                weights = [x[2] for x in valid]
                ridge_slow_rows.append({
                    "slow_window_days": window, "lambda": lam, "k": k,
                    "n_turbines": len(valid), "n_days": int(sum(weights)),
                    "cv_mae": float(np.average([x[0] for x in valid], weights=weights)) if valid else np.nan,
                    "cv_rmse": float(np.sqrt(np.average([x[1] ** 2 for x in valid], weights=weights))) if valid else np.nan,
                })
    display(pd.DataFrame(ridge_slow_rows).round(4))
    print("Slow-background Ridge is diagnostic only; no b, boundary or prediction was modified.")

,slow_window_days,lambda,k,n_turbines,n_days,cv_mae,cv_rmse
0,60,0.50,1.5,5,3595,0.0792,0.1136
1,60,0.50,2.0,5,3595,0.0785,0.1126
2,60,0.50,3.0,5,3595,0.0771,0.1107
3,60,0.75,1.5,5,3595,0.0789,0.1131
4,60,0.75,2.0,5,3595,0.0781,0.1120
5,60,0.75,3.0,5,3595,0.0765,0.1099
6,60,1.00,1.5,5,3595,0.0785,0.1127
7,60,1.00,2.0,5,3595,0.0776,0.1115
8,60,1.00,3.0,5,3595,0.0759,0.1091
9,90,0.50,1.5,5,3595,0.0784,0.1130


Slow-background Ridge is diagnostic only; no b, boundary or prediction was modified.


## Appendix E. Event-level physics corroboration with fixed nuisance

Fit one shared nuisance Ridge only on days far from frozen boundaries, without a physics term. Then residualize power and score physical candidates only at condition-matched boundary windows. This is diagnostic only; it does not alter RRS predictions.

$$z_i(d)=X_i(d)\gamma+\epsilon_i(d)$$

$$\widetilde z_i(d)=z_i(d)-X_i(d)\hat{\gamma}$$

$$E_{k,\lambda}=\left|\Delta\widetilde z_k-\Delta L^{phys}_{k,\lambda}\right|$$

In [16]:
EVENT_K = 2.0
EVENT_EXCLUDE_DAYS = 2 * POWER_WINDOW_DAYS
EVENT_BOOTSTRAPS = 2000
EVENT_NUISANCE_COLUMNS = ["ws", "ws2", "sin_wd", "cos_wd", "bg"]

if not full_fleet_ready:
    print("Event-level physics audit skipped: complete context/test fleet is required.")
else:
    def _event_power_frame(turbine):
        power = relative_power[turbine]
        observed = power.groupby(power.index.normalize()).median().rename("z")
        base = binned[turbine]
        ws = base["WindSpeed"].resample("D").median().rename("ws")
        wd = base["WindDir"].resample("D").median().rename("wd")
        bg = stable_clean[turbine]["observables"]["background_residual_smooth"].rename("bg")
        frame = pd.concat([observed, ws, wd, bg], axis=1, sort=False).dropna()
        frame["ws2"] = frame["ws"] ** 2
        frame["sin_wd"] = np.sin(np.deg2rad(frame["wd"]))
        frame["cos_wd"] = np.cos(np.deg2rad(frame["wd"]))
        return frame

    def _far_from_boundaries(turbine, index):
        table = stable_clean[turbine]["boundaries"]
        active = "prediction_active" if "prediction_active" in table else "accepted"
        dates = [pd.Timestamp(x).normalize() for x in table.index[table[active].fillna(False)]] if len(table) else []
        mask = pd.Series(True, index=index)
        for date in dates:
            mask &= (np.abs((index - date).days) > EVENT_EXCLUDE_DAYS)
        return mask

    event_frames = {t: _event_power_frame(t) for t in audit_ids}
    nuisance_parts = [frame.loc[_far_from_boundaries(t, frame.index)] for t, frame in event_frames.items()]
    nuisance = pd.concat(nuisance_parts, axis=0, sort=False).dropna(subset=["z", *EVENT_NUISANCE_COLUMNS])
    if len(nuisance) < 100:
        print("Event-level physics audit skipped: too few stable nuisance days.")
    else:
        x = nuisance[EVENT_NUISANCE_COLUMNS].to_numpy(float)
        x_mean, x_scale = x.mean(axis=0), x.std(axis=0)
        x_scale[x_scale == 0.0] = 1.0
        x = np.column_stack([np.ones(len(x)), (x - x_mean) / x_scale])
        y = nuisance["z"].to_numpy(float)
        penalty = np.eye(x.shape[1]) * RIDGE_ALPHA
        penalty[0, 0] = 0.0
        nuisance_coef = np.linalg.solve(x.T @ x + penalty, x.T @ y)

        residualized = {}
        for turbine, frame in event_frames.items():
            out = frame.copy()
            xx = out[EVENT_NUISANCE_COLUMNS].to_numpy(float)
            xx = np.column_stack([np.ones(len(xx)), (xx - x_mean) / x_scale])
            out["z_tilde"] = out["z"] - xx @ nuisance_coef
            residualized[turbine] = out

        def _matched_change(frame, date, value_col):
            date = pd.Timestamp(date).normalize()
            pre = frame.loc[(frame.index >= date - pd.Timedelta(days=POWER_WINDOW_DAYS)) & (frame.index < date)].copy()
            post = frame.loc[(frame.index >= date) & (frame.index < date + pd.Timedelta(days=POWER_WINDOW_DAYS))].copy()
            for part in (pre, post):
                part["ws_bin"] = np.floor(part["ws"])
                part["wd_sector"] = np.floor((part["wd"] % 360.0) / 30.0)
            left = pre.groupby(["ws_bin", "wd_sector"])[value_col].median().rename("pre")
            right = post.groupby(["ws_bin", "wd_sector"])[value_col].median().rename("post")
            matched = pd.concat([left, right], axis=1, join="inner").dropna()
            if len(matched) < 2:
                return np.nan, len(matched)
            return float((matched["post"] - matched["pre"]).mean()), len(matched)

        event_rows = []
        for turbine, frame in residualized.items():
            table = stable_clean[turbine]["boundaries"]
            active = "prediction_active" if "prediction_active" in table else "accepted"
            dates = table.index[table[active].fillna(False)] if len(table) else []
            for date in dates:
                date = pd.Timestamp(date).normalize()
                observed_change, n_cells = _matched_change(frame, date, "z_tilde")
                row = {"turbine": turbine, "boundary": date.date().isoformat(),
                       "n_matched_cells": n_cells,
                       "observed_residualized_change": observed_change}
                for lam in LAMBDA_POWER:
                    prediction = predictions_power[lam][turbine]["prediction"]
                    y_pre = _window_median(prediction, date, "pre")
                    y_post = _window_median(prediction, date, "post")
                    phys = _predicted_dlog_power(y_pre, y_post, EVENT_K)
                    row[f"phys_lam{lam:g}"] = phys
                    row[f"abs_error_lam{lam:g}"] = abs(observed_change - phys) if np.isfinite(observed_change) and np.isfinite(phys) else np.nan
                event_rows.append(row)
        event_audit_strict = pd.DataFrame(event_rows)
        display(event_audit_strict.round(4))

        error_cols = [f"abs_error_lam{lam:g}" for lam in LAMBDA_POWER]
        valid = event_audit_strict.dropna(subset=["observed_residualized_change", *error_cols]).copy()
        score_rows = []
        if len(valid):
            errors = valid[error_cols].to_numpy(float)
            rng = np.random.default_rng(20260917)
            draws = rng.integers(0, len(errors), size=(EVENT_BOOTSTRAPS, len(errors)))
            boot_scores = np.mean(errors[draws], axis=1)
            winners = np.argmin(boot_scores, axis=1)
            for j, lam in enumerate(LAMBDA_POWER):
                score_rows.append({"lambda": lam, "k": EVENT_K, "n_events": len(valid),
                                   "event_mae": float(errors[:, j].mean()),
                                   "bootstrap_win_fraction": float(np.mean(winners == j))})
        display(pd.DataFrame(score_rows).round(4))
        print("Nuisance was fit once on stable days; lambda was scored only at matched frozen boundaries.")

,turbine,boundary,n_matched_cells,observed_residualized_change,phys_lam0.5,abs_error_lam0.5,phys_lam0.75,abs_error_lam0.75,phys_lam1,abs_error_lam1
0,PPP_WTG13,2023-07-27,8,0.0607,0.0091,0.0515,0.0111,0.0496,0.0131,0.0476
1,PPP_WTG17,2023-10-15,3,0.0683,-0.0024,0.0707,-0.0030,0.0713,-0.0036,0.0719
2,PPP_WTG17,2024-01-07,7,0.2736,0.0167,0.2569,0.0191,0.2545,0.0215,0.2520
3,SSS_WTG06,2023-05-28,2,-0.0201,-0.0227,0.0026,-0.0266,0.0065,-0.0307,0.0106
4,SSS_WTG06,2023-09-24,6,0.1796,0.0241,0.1555,0.0281,0.1515,0.0323,0.1473


,lambda,k,n_events,event_mae,bootstrap_win_fraction
0,0.50,2.0,5,0.1075,0.2565
1,0.75,2.0,5,0.1067,0.0095
2,1.00,2.0,5,0.1059,0.7340


Nuisance was fit once on stable days; lambda was scored only at matched frozen boundaries.
